In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras as K
from tensorflow.keras.layers import Conv2D, Dense, MaxPooling2D, Flatten, BatchNormalization, Dropout, Activation
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. ПУТЬ
BASE_DIR = 'bluzy_and_bryuki_filtered'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR = os.path.join(BASE_DIR, 'validation')

# Размеры
IMG_HEIGHT = 46
IMG_WIDTH = 66
# Размер батча
BATCH_SIZE = 32
# Количество эпох
EPOCHS = 20

# 2. ПОДСЧЕТ ФАЙЛОВ
try:
    totalTrain = len(os.listdir(os.path.join(TRAIN_DIR, 'bluzy'))) + \
                 len(os.listdir(os.path.join(TRAIN_DIR, 'bryuki')))
    totalVal = len(os.listdir(os.path.join(VAL_DIR, 'bluzy'))) + \
               len(os.listdir(os.path.join(VAL_DIR, 'bryuki')))
    print(f"Найдено изображений для обучения: {totalTrain}")
    print(f"Найдено изображений для проверки: {totalVal}")
except FileNotFoundError:
    print("ОШИБКА: Папка 'bluzy_and_bryuki_filtered' не найдена!")
    print("Убедитесь, что она лежит рядом с файлом .ipynb")

# 3. ГЕНЕРАТОРЫ ДАННЫХ
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

print("Загрузка данных...")
trainingData = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH), # (46, 66)
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True
)

validationData = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print("Метки классов:", trainingData.class_indices)

# 4. СОЗДАНИЕ МОДЕЛИ
K.backend.clear_session()

model = K.models.Sequential([
    # Слой 1
    Conv2D(16, (3, 3), padding='same', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    Activation('relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Слой 2
    Conv2D(32, (3, 3), padding='same'),
    Activation('relu'),
    MaxPooling2D(pool_size=(2, 2)),

    # Слой 3
    Conv2D(64, (3, 3), padding='same'),
    Activation('relu'),
    MaxPooling2D(pool_size=(2, 2)),
    
    # Слой 4 (Последний сверточный)
    Conv2D(128, (3, 3), padding='same'),
    Activation('relu'),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    
    # Полносвязные слои
    Dense(512, activation='relu'),
    Dropout(0.5),
    
    # Выходной слой (2 нейрона, softmax)
    Dense(2, activation='softmax') 
], name='bluzy-vs-bryuki')

# Компиляция
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Подходит для меток 0 и 1
    metrics=['accuracy']
)

model.summary()

# 5. ЗАПУСК ОБУЧЕНИЯ
history = model.fit(
    trainingData,
    steps_per_epoch=totalTrain // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=validationData,
    validation_steps=totalVal // BATCH_SIZE
)

# 6. СОХРАНЕНИЕ МОДЕЛИ
model.save('bluzy_bryuki_model.h5')
print("Модель успешно сохранена как bluzy_bryuki_model.h5")